# Assumptions of Linear Regression

For the predictions and statistical inferences of a Linear Regression model to be valid and reliable, the underlying data and residuals must satisfy **six core assumptions**. If these assumptions are violated, the model's coefficients might be biased, standard errors could be incorrect (leading to invalid p-values), and predictions might be unstable.

---

## Summary of the 6 Core Assumptions

| Assumption | Description | Diagnostic Tool | How to Fix |
| :--- | :--- | :--- | :--- |
| **1. Linearity** | Linear relationship between features ($X$) and target ($Y$). | • Scatter plots<br>• Residuals vs. Fitted plot | • Log / square root transformations<br>• Polynomial Regression |
| **2. Independence of Residuals** | Errors of different observations are independent. | • Durbin-Watson statistic (ideal $\approx 2$)<br>• Residuals vs. Time plot | • Use time-series models (ARIMA)<br>• Generalized Least Squares (GLS) |
| **3. Homoscedasticity** | Constant variance of residuals across all feature levels. | • Residuals vs. Fitted plot (funnel shape = bad) | • Transform target variable ($
\log(Y)$)<br>• Weighted Least Squares (WLS) |
| **4. Normality of Residuals** | Errors follow a normal distribution. | • Q-Q Plot<br>• Histogram / KDE of residuals<br>• Shapiro-Wilk test | • Box-Cox or Log transformation of target |
| **5. No Multicollinearity** | Predictors are not highly correlated with each other. | • Correlation Heatmap<br>• Variance Inflation Factor (VIF > 5/10) | • Remove one of the correlated features<br>• PCA<br>• Ridge / Lasso regression |
| **6. No Influential Outliers** | No data points unduly skew the regression line. | • Cook's Distance ($> 4/n$ or $> 1$)<br>• Residuals vs. Leverage plot | • Remove data entry errors<br>• Use Robust Regression (RANSAC, Huber) |

---


## Code Setup & Data Generation

To understand these assumptions, we will generate synthetic datasets that deliberately violate or satisfy them, then build diagnostic plots to identify the issues.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


## 1. Linearity

> **Definition:** The relationship between the independent variable(s) and the dependent variable must be linear. If the true relationship is curve-linear, a straight line will underfit the data.

### Diagnostics:
- **Residuals vs. Fitted plot:** In a good model, residuals should be randomly scattered around the $y=0$ line. A curved pattern (like a U-shape) indicates a violation of linearity.

Let's visualize a curved relationship trained using standard linear regression.

In [ ]:
np.random.seed(42)
x = np.random.uniform(-3, 3, 100)
# Non-linear quadratic relationship
y_non_linear = x**2 + np.random.normal(0, 1, 100)

# Fit linear model
lr = LinearRegression()
lr.fit(x.reshape(-1, 1), y_non_linear)
y_pred = lr.predict(x.reshape(-1, 1))
residuals = y_non_linear - y_pred

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot with regression line
axes[0].scatter(x, y_non_linear, color='#3498db', alpha=0.7, label='Data')
axes[0].plot(np.sort(x), lr.predict(np.sort(x).reshape(-1, 1)), color='#e74c3c', linewidth=2.5, label='Regression Line')
axes[0].set_title('Curve-linear Data fitted with Straight Line')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].legend()

# Residual plot
axes[1].scatter(y_pred, residuals, color='#2ecc71', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs. Fitted Plot (Violation Visible)')
axes[1].set_xlabel('Fitted Values (Predicted)')
axes[1].set_ylabel('Residuals (Actual - Predicted)')

plt.tight_layout()
plt.show()


## 2. Homoscedasticity (Equal Variance)

> **Definition:** The variance of the residual errors must be constant across all values of the independent variables. If the variance changes (typically expanding as $X$ grows), it is called **Heteroscedasticity**.

### Impact:
- Standard errors will be calculated incorrectly, making statistical significance tests (p-values) untrustworthy.

### Diagnostics:
- **Residuals vs. Fitted plot:** In heteroscedasticity, the residuals form a **funnel or cone shape** rather than a uniform horizontal band.

In [ ]:
np.random.seed(42)
x = np.linspace(1, 10, 150)
# Heteroscedastic error: error size increases with x
error = np.random.normal(0, 0.4 * x, 150)
y_hetero = 2 * x + error

# Fit model
lr = LinearRegression()
lr.fit(x.reshape(-1, 1), y_hetero)
y_pred = lr.predict(x.reshape(-1, 1))
residuals = y_hetero - y_pred

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(x, y_hetero, color='#3498db', alpha=0.7, label='Data')
axes[0].plot(x, y_pred, color='#e74c3c', linewidth=2.5, label='Regression Line')
axes[0].set_title('Data with Heteroscedasticity')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].legend()

# Residual plot showing funnel shape
axes[1].scatter(y_pred, residuals, color='#2ecc71', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title('Funnel-shaped Residual Plot (Heteroscedasticity)')
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()


## 3. Normality of Residuals

> **Definition:** The residuals (errors) must be normally distributed. This is crucial for hypothesis testing (e.g., verifying if coefficients are significantly different from zero using p-values).

### Diagnostics:
- **Normal Q-Q Plot (Quantile-Quantile Plot):** Plots the quantiles of the residuals against the quantiles of a standard normal distribution. If normal, points lie along the straight 45-degree diagonal line.
- **Histogram / KDE:** Visually checks if residuals show a bell-shaped curve.

In [ ]:
np.random.seed(42)
x = np.random.uniform(1, 10, 150)
# Non-normal skewed error terms (e.g. exponential distribution)
error = np.random.exponential(scale=2, size=150) - 2
y_non_normal = 1.5 * x + error

# Fit model
lr = LinearRegression()
lr.fit(x.reshape(-1, 1), y_non_normal)
residuals = y_non_normal - lr.predict(x.reshape(-1, 1))

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE / Histogram
sns.histplot(residuals, kde=True, ax=axes[0], color='#3498db')
axes[0].set_title('Residuals Distribution (Skewed / Non-Normal)')
axes[0].set_xlabel('Residual value')

# Q-Q Plot
sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].get_lines()[1].set_color('#e74c3c')  # Reference line red
axes[1].set_title('Normal Q-Q Plot (Deviation from diagonal = Non-Normal)')

plt.tight_layout()
plt.show()


## 4. No Multicollinearity

> **Definition:** Independent variables should not be highly correlated with each other. If predictors are collinear, it becomes difficult for the model to isolate the individual effect of each feature on the target.

### Diagnostics:
- **Correlation Heatmap:** Visual identification of highly correlated features ($|r| > 0.8$).
- **Variance Inflation Factor (VIF):** Measures how much the variance of an estimated regression coefficient is inflated due to collinearity.

$$\text{VIF}_i = \frac{1}{1 - R_i^2}$$

| VIF Value | Interpretation |
| :--- | :--- |
| **VIF = 1** | No correlation |
| **1 < VIF < 5** | Moderate correlation (Acceptable) |
| **VIF > 5 or 10** | High correlation (Severe Multicollinearity - must be addressed) |

In [ ]:
# Generate collinear features
np.random.seed(42)
cgpa = np.random.uniform(5, 10, 100)
# IQ is highly correlated with CGPA
iq = cgpa * 12 + np.random.normal(0, 2, 100)
# Age is independent
age = np.random.randint(18, 25, 100)

# Combine into a DataFrame
df_coll = pd.DataFrame({
    'CGPA': cgpa,
    'IQ': iq,
    'Age': age
})

# Calculate VIF for each feature
# We add a constant column since VIF calculation requires intercept term
X_vif = sm.add_constant(df_coll)
vif_data = pd.DataFrame()
vif_data["Feature"] = df_coll.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(df_coll.shape[1])]

print("VIF scores calculated:")
print("="*35)
print(vif_data.to_string(index=False))
print("="*35)
print("Notice: CGPA and IQ have extremely high VIF scores indicating severe multicollinearity!")


## 5. No Influential Outliers

> **Definition:** Regression lines are highly sensitive to outliers. A single point placed far from the main dataset (especially with extreme feature values) can pull the regression line towards itself, distorting overall model parameters.

### Diagnostics:
- **Cook's Distance:** Measures the effect of deleting a given observation. Points with Cook's distance $> 4/n$ (where $n$ is sample size) or $> 1.0$ are considered highly influential outliers.

In [ ]:
np.random.seed(42)
x = np.random.uniform(1, 5, 50)
y = 2 * x + np.random.normal(0, 0.5, 50)

# Introduce a single influential outlier (High Leverage point)
x_out = np.append(x, [12.0])
y_out = np.append(y, [4.0])  # Normal trend would put y around 24

# Fit model with and without outlier
lr_clean = LinearRegression().fit(x.reshape(-1, 1), y)
lr_outlier = LinearRegression().fit(x_out.reshape(-1, 1), y_out)

# Plotting the impact
plt.figure(figsize=(10, 6))
plt.scatter(x, y, color='#3498db', alpha=0.7, label='Clean Data')
plt.scatter([12.0], [4.0], color='#e74c3c', s=100, marker='X', label='High Leverage Outlier')

# Plot lines
x_line = np.linspace(1, 13, 100).reshape(-1, 1)
plt.plot(x_line, lr_clean.predict(x_line), color='#2ecc71', linewidth=2.5, linestyle='--', label='Clean Regression Line')
plt.plot(x_line, lr_outlier.predict(x_line), color='#e74c3c', linewidth=2.5, label='Distorted Regression Line')

plt.title("Impact of a Single High-Leverage Outlier on Regression Line", fontsize=13, fontweight='bold')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.show()


## Summary Checklist for Regression Analysis

When building a Linear Regression model, perform this workflow to ensure assumptions are satisfied:

1. **Check Linearity:** Plot inputs vs. output. Run `y_pred` vs. `residuals` scatter. If curved, apply feature transformation.
2. **Check Homoscedasticity:** Inspect the residual scatter plot. Look for constant width. If funnel-shaped, apply log transformation on target variable $Y$.
3. **Check Normality:** Generate a Q-Q plot of the model residuals. Ensure points follow the diagonal.
4. **Check Multicollinearity:** Calculate Variance Inflation Factors (VIF) on independent features. Remove any feature with a VIF score $> 5$.
5. **Check Outliers:** Run Cook's distance diagnostic. Inspect points with high distances to check for recording errors.

---

> 💡 **Pro Tip:** In Python, the `statsmodels` library provides `sm.OLS` which includes detailed diagnostic printouts containing statistics like **Jarque-Bera** (for Normality) and **Durbin-Watson** (for Independence) directly inside `model.summary()`!